# 03 - PV hosting capacity

## Objective

Understand hosting capacity as a search result tied to a declared criterion, then compare a small direct OpenDSS bracket with the CEPT public CLI result.

## Source, assumptions, and units

The source is the IEEE13 feeder bundled in the installed CEPT wheel. The demonstrator adds a three-phase, unity-power-factor PV at bus `675`. The overvoltage criterion is `v_max = 1.05 pu`; candidate PV sizes are in kW. These values are declared teaching inputs, not an interconnection study or a universal hosting-capacity limit.

## Prediction

Increasing PV active power should increase the feeder's maximum unregulated voltage in this setup. The 1000 kW and 2000 kW direct points should bracket the criterion, and the CEPT result should report a bus-specific value in that bracket.

## Action

Run the direct OpenDSS bracket, then stream `cept study demo hosting-capacity` into this notebook. The CLI writes an exact run directory instead of leaving the result only in memory.

## Verification

Read the actual baseline and hosting-capacity rows from `results.json`, run `cept study verify`, and assert the declared bracket and criterion.

## Interpretation

The returned `hc_kw` is criterion-specific and model-specific. It is not utility approval, a thermal rating, a protection result, or a project-validation claim.

## Exercise

Change `DIRECT_SIZES_KW` or the displayed criterion only after stating a new prediction. Rerun from a restarted kernel and keep the exact input and run path with the resulting table.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
# @title Setup — run first, then read the results below
import hashlib
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import urllib.parse
import urllib.request
from importlib.resources import files
from pathlib import Path
import html
from IPython.display import HTML, display

# Notebook workspace root, captured before any solver call: OpenDSS
# DataPath changes the process working directory, so later cells must not
# rely on Path.cwd().
WORKSPACE = Path.cwd()

DEFAULT_WHEEL_URL = 'https://github.com/sarutesri/cept-studio-edu/releases/download/v0.2.0-edu.1/cept_power_studio-0.2.0.dev0-py3-none-any.whl'
DEFAULT_WHEEL_SHA256 = 'c7e609a1d9cc85b322bfb615f0c796c7ea5c43815b197289eb555786964478bc'
configured_url = os.environ.get('CEPT_WHEEL_URL')
WHEEL_URL = (DEFAULT_WHEEL_URL if configured_url is None and importlib.util.find_spec('cept') is None else (configured_url or '')).strip()
WHEEL_SHA256 = os.environ.get('CEPT_WHEEL_SHA256', DEFAULT_WHEEL_SHA256 if WHEEL_URL == DEFAULT_WHEEL_URL else '').strip().lower()
if WHEEL_URL:
    if len(WHEEL_SHA256) != 64 or any(character not in '0123456789abcdef' for character in WHEEL_SHA256):
        raise ValueError('CEPT_WHEEL_SHA256 must be the caller-provided 64-character SHA-256')
    wheel_path = Path.cwd() / Path(urllib.parse.urlparse(WHEEL_URL).path).name
    print(f'Downloading caller-provided wheel: {WHEEL_URL}')
    urllib.request.urlretrieve(WHEEL_URL, wheel_path)
    digest = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if digest != WHEEL_SHA256:
        raise ValueError(f'wheel hash mismatch: expected {WHEEL_SHA256}, got {digest}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)], check=True)
else:
    print('CEPT_WHEEL_URL not supplied; using the existing installed environment.')
CLI_BIN = Path(sys.executable).parent / ('cept.exe' if os.name == 'nt' else 'cept')
if not CLI_BIN.is_file():
    raise RuntimeError('cept launcher not found next to Python; reinstall the pinned public wheel')
CLI = str(CLI_BIN)
print('CLI: cept --version')
print(subprocess.run([CLI, '--version'], capture_output=True, text=True, check=True).stdout.strip())
def run_cli(*arguments, verbose=False):
    # Quiet by default: result tables below are the lesson. Pass verbose=True
    # to stream the full solver-backed receipt instead.
    display_cmd = 'cept ' + shlex.join([str(argument) for argument in arguments])
    command = [CLI, *[str(argument) for argument in arguments]]
    print('$ ' + display_cmd, flush=True)
    if verbose:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd())
        lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            lines.append(line)
        returncode = process.wait()
        output = ''.join(lines)
    else:
        completed = subprocess.run(command, capture_output=True, text=True, cwd=Path.cwd())
        returncode, output = completed.returncode, completed.stdout + completed.stderr
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ['cept', *[str(argument) for argument in arguments]], output=output[-4000:])
    print('\u2192 exit 0', flush=True)
    return json.loads(output) if output.strip().startswith('{') else output

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def show_table(headers, rows):
    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join('---' for _ in headers) + ' |')
    for row in rows:
        print('| ' + ' | '.join(str(value) for value in row) + ' |')

def cards(items, title='CEPT Studio'):
    blocks = []
    for label, value, note in items:
        blocks.append(f'''<div style="flex:1;min-width:180px;border:1px solid #d9dee8;border-radius:14px;padding:14px 16px;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.05)"><div style="font-size:12px;color:#667085;text-transform:uppercase;letter-spacing:.04em">{html.escape(str(label))}</div><div style="font-size:22px;font-weight:700;margin:4px 0;color:#182230">{html.escape(str(value))}</div><div style="font-size:12px;color:#667085">{html.escape(str(note))}</div></div>''')
    display(HTML(f'''<div style="font-family:Inter,Arial,sans-serif;margin:10px 0 18px"><div style="font-size:18px;font-weight:700;margin-bottom:9px">{html.escape(title)}</div><div style="display:flex;gap:10px;flex-wrap:wrap">{"".join(blocks)}</div></div>'''))


MASTER_DSS = Path(str(files('cept').joinpath('testsystems', 'ieee13', 'IEEE13Nodeckt.dss')))
DIRECT_SIZES_KW = (0, 1000, 2000)
V_MAX_PU = 1.05


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0


In [2]:
import opendssdirect as dss

def load_base():
    dss.Basic.ClearAll()
    dss.Basic.DataPath(str(MASTER_DSS.parent))
    dss.Text.Command(f'Redirect \"{MASTER_DSS}\"')
    dss.Text.Command('CalcVoltageBases')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()

def max_unregulated_voltage():
    excluded = {'sourcebus', '650', 'rg60'}
    names = dss.Circuit.AllNodeNames()
    values = dss.Circuit.AllBusMagPu()
    return max(value for name, value in zip(names, values) if name.split('.')[0].lower() not in excluded)

direct_sweep = {}
for kw in DIRECT_SIZES_KW:
    load_base()
    dss.Text.Command(f'New PVSystem.lesson_pv phases=3 bus1=675.1.2.3 kV=4.16 kVA={max(kw, 1)} Pmpp={kw} irradiance=1 pf=1 %cutin=0.05 %cutout=0.05')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()
    direct_sweep[kw] = max_unregulated_voltage()
show_table(['PV size', 'maximum unregulated voltage', 'unit'], [(kw, value, 'pu') for kw, value in direct_sweep.items()])
assert direct_sweep[1000] < V_MAX_PU < direct_sweep[2000]


| PV size | maximum unregulated voltage | unit |
| --- | --- | --- |
| 0 | 1.0426313303211827 | pu |
| 1000 | 1.0476066810380371 | pu |
| 2000 | 1.0521988003106502 | pu |


In [3]:
RUN_DIR = WORKSPACE / 'runs' / '03-hosting-capacity'
run_summary = run_cli('study', 'demo', 'hosting-capacity', '--network', 'ieee13', '--out', RUN_DIR, '--force')
verify_summary = run_cli('study', 'verify', RUN_DIR)
results = read_json(RUN_DIR / 'results.json')
hosting = results['hosting_capacity']
show_table(['source', 'field', 'value', 'unit'], [('CEPT results.json', 'criterion', hosting['criterion'], 'text'), ('CEPT results.json', 'v_max', hosting['v_max_pu'], 'pu'), ('CEPT results.json', 'baseline_v_max', hosting['baseline_v_max_pu'], 'pu')])
show_table(['bus', 'hosting capacity', 'limiting condition', 'voltage at capacity', 'units'], [(row['bus'], row['hc_kw'], row['limit'], row['v_at_hc'], 'kW / pu') for row in hosting['items']])
item = next(row for row in hosting['items'] if row['bus'].lower() == '675')

cards([
    ('CEPT run', run_summary['status'], 'solver-backed run artifacts'),
    ('Verify', str(verify_summary['passed']), 'receipt integrity and convergence'),
    ('Bus 675 capacity', f"{item['hc_kw']} kW", 'declared-criterion search'),
    ('Limit', str(item['limit']), 'binding condition at capacity'),
], title='3 \u00b7 Hosting-capacity result')
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert abs(direct_sweep[0] - hosting['baseline_v_max_pu']) < 1e-3
assert 1000 <= item['hc_kw'] <= 2000
assert hosting['criterion'] == 'overvoltage' and hosting['v_max_pu'] == V_MAX_PU


$ cept study demo hosting-capacity --network ieee13 --out '<notebook-workspace>\runs\03-hosting-capacity' --force


→ exit 0


$ cept study verify '<notebook-workspace>\runs\03-hosting-capacity'


→ exit 0


| source | field | value | unit |
| --- | --- | --- | --- |
| CEPT results.json | criterion | overvoltage | text |
| CEPT results.json | v_max | 1.05 | pu |
| CEPT results.json | baseline_v_max | 1.0426 | pu |
| bus | hosting capacity | limiting condition | voltage at capacity | units |
| --- | --- | --- | --- | --- |
| 633 | 4498.9 | overvoltage | 1.05 | kW / pu |
| 671 | 3088.7 | overvoltage | 1.05 | kW / pu |
| 692 | 3088.7 | overvoltage | 1.05 | kW / pu |
| 675 | 1507.6 | overvoltage | 1.05 | kW / pu |
| 670 | 3669.1 | overvoltage | 1.05 | kW / pu |
| 632 | 4355.8 | overvoltage | 1.05 | kW / pu |
| 680 | 4309.1 | overvoltage | 1.05 | kW / pu |


The rows above are read from solver-backed direct OpenDSS values and the exact CEPT `results.json`. The bracket supports a teaching interpretation of this demonstrator only. A real hosting-capacity decision needs source-bound ratings, scenarios, protection and control settings, applicable criteria, and reviewer acceptance.